<a href="https://colab.research.google.com/github/GabCAD92/Machine-learning-tasks/blob/main/GAB_OWOLABI_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Machine Learning
## Supervised Learning: GANs practice

This lesson implements __Generative Adversarial Networks (GANs)__ for __fake face images generation__ using __Keras__.


### Professor:

<img src="https://www.sorocaba.unesp.br/Home/Graduacao/EngenhariadeControleeAutomacao/alexandre/alex_marta1_small.jpg" width="100" style="float: left; margin-right: 5px;" border="10px" />

  __Prof. Dr. Alexandre da Silva Simões__ <br>
  Control and Automation Engineering Department (DECA) <br>
  Institute of Science and Technology<br>
  São Paulo State University (Unesp) <br>
  Campus Sorocaba <br>
  www.sorocaba.unesp.br/professor/assimoes

<br/>




____

# Table of Contents


1. Introduction <br>
  1.1. CelebA dataset <br>
	1.2. Loading from zip file<br>
	1.3. Limited access to files in Google Drive<br>
2. GAN implementation <br>
	2.1. Configurations <br>
	2.2. Load images into RAM <br>
	2.3. Evaluation <br>
	2.4. Generator Architecture <br>
	2.5. Discriminator Architecture <br>
	2.6. Other G-D architectures <br>
	2.7. Loss functions <br>
	2.8. Training <br>
	2.9 Main Loop <br>
	2.10. View history <br>
	2.11. Generating faces <br>
3. Next steps <br>
 <br>


# 1. Introduction

## 1.1. CelebFaces Attributes Dataset (CelebA)

<center>
  <img src="https://drive.google.com/uc?id=1SCilqQHsrVC3EGANQGBxBcOTV90xqXy6" width="500" style="float: left; margin-right: 5px;" border="0px" />
</center>


__CelebFaces Attributes Dataset__ (__CelebA__) contains __faces of celebrities__. It is one of the most popular datasets in computer vision and machine learning fields, especially used in tasks such as image generation, facial recognition, physical attribute detection, and evaluation of generative models such as GANs. The dataset is available at: http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html


* Number of images: __202,599 faces__
* Number of identities: __10,177 celebrities__
* Number of attributes: 40 binary attributes per image (such as "smiling", "eyeglasses")
* Number of landmarks 5 facial landmarks per image (eyes, nose, corners of mouth)
* Original resolution of images varies, usually around 178x218 pixels
* Resized versions often resized to 64×64, 128×128 or 256×256
JPEG format

## 1.2. Loading from zip file

In our experiment we will use a file callle `img_align_celeba.zip`, which contains only the celebrity images, one in each file.

* Images width: 178 pixels
* Images height: 218 pixels
* Image channels: 3 (RGB)
* Notes:
  * These images have been pre-aligned based on facial landmarks, hence the name img_align_celeba;
  * When used in most machine learning tasks (especially with GANs), they are often:
    * Cropped to a square (typically the central 178×178 region);
    * Resized to 64×64, 128×128, or 256×256, depending on the model architecture and available computational resources.

## 1.3. Limited access to files in Google Drive

Since Google limits the number of accesses to files in Googre Drive, this code was develop to __extract all images from the zip file directly to RAM__, whithou loading or saving individual image files.
<br>
<br>




# 2. GAN Implementation

Let's implemente our GAN to generate artificial celebrities faces...

## 2.1. Configurations

Here are our main constants and libraries...<br>
As a reference, this is the configuraton of the **original DCGAN** (RADFORD et. al., 2015): <br>

  * Architecture:
    * Generator: Dense → reshape → Conv2DTranspose layers (256, 512, 1024 filters), BatchNorm, ReLU → final Conv2DTranspose to 3-channel output with tanh
    * Discriminator: Conv2D (64, 128, 256, etc.), BatchNorm, LeakyReLU → Dense(1)
  * Training parameters:
    * LEARNING_RATE_G = 0.0002
    * LEARNING_RATE_D = 0.0002
    * Otimization: Adam (BETA_1 = 0.5, BETA_2 = 0.999)
    * BATCH_SIZE = 128
    * NOISE_DIM = 100


In [1]:
# -------------------------------------------------
# GAN implementation for generation of fake faces
# based on CelebA dataset in zip file
# -------------------------------------------------

# Images parameters
IMAGE_WIDTH = 64           # Width of input and generated images (in pixels)
IMAGE_HEIGHT = 64          # Height of input and generated images (in pixels)
IMAGE_CHANNELS = 3         # Number of color channels (RGB = 3)
NUM_IMAGES = 50000         # Number of images loaded from the zip file (until 202,599)

# Pre-training parameters
BUFFER_SIZE = 10000        # Size of the shuffle buffer for the dataset pipeline
BATCH_SIZE = 128           # Number of images per training batch (when higher can make the learnin smoother) (ref:256)
NOISE_DIM = 128            # Dimensionality of the input noise vector for the generator

# Architecture
MODEL_TYPE = 'DCGAN'       # GAN architecture type; options: DCGAN, PGGAN, WGAN, StyleGAN
# G:
NORM_TYPE_G = 'BATCH'      # Normalization in G. Select between: {BATCH, LAYER, NONE}
USE_RELU_G = 1             # Use ReLu in G? {1: YES, 0: NO}
USE_DROPOUT_G = 0          # Use DropOut in G? {1: YES, 0: NO}
DROPOUT_G = 0.0            # Dropout rate in G (beginning of learning: 0-0.1, instable/collapse: 0-0.2, stable: 0.2-0.4, with batchnorm: <=0.3)
# D:
NORM_TYPE_D = 'BATCH'      # Normalization in D. Select between: {BATCH, NONE}
USE_RELU_D = 1             # Use ReLu in D? {1: YES, 0: NO}
USE_DROPOUT_D = 0          # Use DropOut in D? {1: YES, 0: NO}
DROPOUT_D = 0.0            # Dropout rate in D (avoid underfitting/can generate overfitting: <0.2, most common: 0.2-0.3, D too strong: 0.4-0.5)

# Training parameters
MAX_EPOCHS = 100           # Total number of epochs to train the GAN
EARLY_STOP_PATIENCE = 5   # Number of epochs to wait for FID improvement before early stopping
LEARNING_RATE_G = 1e-3     # Learning rate for the generator optimizer (6e-4)
LEARNING_RATE_D = 2e-4     # Learning rate for the discriminator optimizer (4e-5)
NUM_STEPS_G = 1            # Number of generator updates per iteration
NUM_STEPS_D = 1            # Number of discriminator updates per iteration
BETA_1 = 0.5               # Exponential decay rate for the first moment estimates (momentum term) in Adam optimizer
BETA_2 = 0.999             # Exponential decay rate for the second moment estimates (variance term) in Adam optimizer

# Visual output setup
VIS_INTERVAL = 1           # Interval (in epochs) to visualize outputs and metrics
TEXT_SIZE = 11             # Font size for plot labels and titles
COLOR_G = (0.58, 0.71, 0.82)# Color for generator plot
COLOR_D = (1, 0.6 , 0.4)    # COlor for discriminator plot

# Load/save parameters
LOAD_BEST_MODEL = 0        # Whether to load a previously saved model (1 = yes, 0 = no)
SAVE_HISTORY = 1           # Save training history all epochs to file

# CelebA dataset
ZIP_PATH = "/content/drive/MyDrive/img_align_celeba.zip"
# Path to the CelebA dataset ZIP file containing aligned face images

# Required libraries

import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from keras.models import load_model
from tensorflow.keras.optimizers import Adam
from matplotlib.ticker import LogLocator
from sklearn.metrics import pairwise_distances
from tensorflow.keras import layers, Input, Sequential, regularizers
from google.colab import drive
from tqdm.auto import tqdm
from scipy import linalg
import logging
import pickle
logging.getLogger('tensorflow').setLevel(logging.ERROR)



## 2.2. Load images into RAM

Here, in order to avoid problems with Google Drive, let's load our images to the RAM...

In [2]:

# Load images from ZIP into RAM
def load_celeba_images(zip_path, max_images=None):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        img_files = [f for f in zip_ref.namelist() if f.endswith('.jpg')]
        if max_images:
            img_files = img_files[:max_images]
        imgs = []
        for f in tqdm(img_files, desc='Loading images'):
            with zip_ref.open(f) as img_file:
                img = tf.io.decode_jpeg(img_file.read(), channels=3)
                img = tf.image.resize(img, [IMAGE_HEIGHT, IMAGE_WIDTH])
                img = (img / 127.5) - 1
                imgs.append(img)
        return tf.data.Dataset.from_tensor_slices(tf.stack(imgs)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)



## 2.3.  Evaluation

Here we will define a method to evaluate our GAN based on:

* FID score
* G and D Gradients
* Precision and Recall
* Generated images analysis

In [3]:

def preprocess_for_fid(images):
    images = tf.image.resize(images, (299, 299))
    images = (images + 1.0) * 127.5
    return tf.keras.applications.inception_v3.preprocess_input(images)


def calculate_fid(act1, act2):
    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2.0 * covmean)

def compute_fid(real_images, fake_images):
    act1 = inception_model(preprocess_for_fid(real_images)).numpy()
    act2 = inception_model(preprocess_for_fid(fake_images)).numpy()
    return calculate_fid(act1, act2)

def compute_precision_recall(real_images, fake_images, percentile=10):
    act1 = inception_model(preprocess_for_fid(real_images)).numpy()
    act2 = inception_model(preprocess_for_fid(fake_images)).numpy()
    dists = pairwise_distances(act1, act2)
    # Usa o percentil inferior como threshold adaptativo
    dynamic_thresh = np.percentile(dists, percentile)
    precision = np.mean(np.min(dists, axis=0) < dynamic_thresh)
    recall = np.mean(np.min(dists, axis=1) < dynamic_thresh)
    return precision, recall

def plot_training_metrics(history):
    epochs = range(1, len(history['gen_loss']) + 1)
    plt.figure(figsize=(10, 4))

    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['gen_loss'], 'o-', label=r'$\mathcal{L}_G$', color=COLOR_G)
    plt.plot(epochs, history['disc_loss'], 'o-', label=r'$\mathcal{L}_D$', color=COLOR_D)
    plt.xlabel('Epoch', fontsize=TEXT_SIZE)
    plt.ylabel('Loss', fontsize=TEXT_SIZE)
    plt.grid(True)
    plt.legend(fontsize=TEXT_SIZE+4)
    plt.title('Loss over Epochs', fontsize=TEXT_SIZE)

    # Evaluation and Gradients plot
    ax2 = plt.subplot(1, 2, 2)
    fid_values = history['FID']
    ax2.plot(epochs, fid_values, 'o-', label='FID')
    ax2.plot(epochs, history['precision'], 'o-', label='Precision', color='y')
    ax2.plot(epochs, history['recall'], 'o-', label='Recall', color='g')

    # Optional: gradient norm lines
    if 'grad_norm_gen' in history:
        ax2.plot(epochs, history['grad_norm_gen'], 'v--', label=r'$\nabla G$', alpha=0.7, color=COLOR_G)
    if 'grad_norm_disc' in history:
        ax2.plot(epochs, history['grad_norm_disc'], 'v--', label=r'$\nabla D$', alpha=0.7, color=COLOR_D)

    ax2.set_xlabel('Epoch', fontsize=TEXT_SIZE)
    ax2.set_ylabel('Score', fontsize=TEXT_SIZE)
    ax2.set_yscale('log')
    ax2.set_ylim(bottom=0.1)

    # Highlight best FID point
    best_fid = min(fid_values)
    best_epoch = epochs[fid_values.index(best_fid)]
    ax2.plot(best_epoch, best_fid, 'o', markerfacecolor='none', markeredgecolor='red', markersize=12, markeredgewidth=2)
    ax2.text(best_epoch, best_fid * 1.2, f"{best_fid:.2f}", color='red', fontsize=9, ha='center')

    # Log ticks
    ax2.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0,), numticks=10))
    ax2.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10)*0.1, numticks=100))
    ax2.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax2.legend(fontsize=TEXT_SIZE-1)
    ax2.set_title('Evaluation Metrics & Gradients', fontsize=TEXT_SIZE)

    plt.tight_layout()
    plt.show()

def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow((predictions[i] * 127.5 + 127.5).numpy().astype(np.uint8))
        plt.axis('off')
    plt.suptitle(f'Epoch {epoch}')
    plt.show()

## 2.4. Generator Architecture

Let's define our __DCGAN__ generator (G) architecture...

In [4]:
def make_generator_dcgan():
    model_layers = [
        Input(shape=(NOISE_DIM,)),
        layers.Dense(4 * 4 * 1024, use_bias=False),
        layers.Reshape((4, 4, 1024)),
    ]

    # Normalization and activation
    if NORM_TYPE_G == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    elif NORM_TYPE_G == 'LAYER':
        model_layers.append(layers.LayerNormalization(axis=[1, 2, 3], epsilon=1e-5))
    if USE_RELU_G == 1:
        model_layers.append(layers.ReLU())
    if USE_DROPOUT_G == 1:
        model_layers.append(layers.Dropout(DROPOUT_G))

    # Conv2DTranspose: 1024 → 512
    model_layers.append(layers.Conv2DTranspose(512, kernel_size=4, strides=2, padding='same', use_bias=False))
    if NORM_TYPE_G == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    elif NORM_TYPE_G == 'LAYER':
        model_layers.append(layers.LayerNormalization(axis=[1, 2], epsilon=1e-5))
    if USE_RELU_G == 1:
        model_layers.append(layers.ReLU())
    if USE_DROPOUT_G == 1:
        model_layers.append(layers.Dropout(DROPOUT_G))

    # 512 → 256
    model_layers.append(layers.Conv2DTranspose(256, kernel_size=4, strides=2, padding='same', use_bias=False))
    if NORM_TYPE_G == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    elif NORM_TYPE_G == 'LAYER':
        model_layers.append(layers.LayerNormalization(axis=[1, 2], epsilon=1e-5))
    if USE_RELU_G == 1:
        model_layers.append(layers.ReLU())
    if USE_DROPOUT_G == 1:
        model_layers.append(layers.Dropout(DROPOUT_G))

    # 256 → 128
    model_layers.append(layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False))
    if NORM_TYPE_G == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    elif NORM_TYPE_G == 'LAYER':
        model_layers.append(layers.LayerNormalization(axis=[1, 2], epsilon=1e-5))
    if USE_RELU_G == 1:
        model_layers.append(layers.ReLU())
    if USE_DROPOUT_G == 1:
        model_layers.append(layers.Dropout(DROPOUT_G))

    # Saída: 128 → 3 (image)
    model_layers.append(layers.Conv2DTranspose(IMAGE_CHANNELS, kernel_size=4, strides=2, padding='same',
                                               use_bias=False, activation='tanh'))

    return Sequential(model_layers)

Let's check out this model using __visualkeras__ ... <br>
If necessary, uncomment the line to install visualkeras package...

In [6]:
# ---------------
# Visualize model
# ---------------

!pip install visualkeras      # Install library in colab, if necessary
import visualkeras
model = make_generator_dcgan()
model.build(input_shape=(None, NOISE_DIM)) # Explicitly build the model
visualkeras.layered_view(model, legend=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/visualkeras/layered.py:231: UserWarning: The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.
  warnings.warn("The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.")


AttributeError: 'Dense' object has no attribute 'output_shape'

For now, let's remove this model from the memory...

In [7]:
del model

## 2.5. Discriminator Architecture

Now let's define our __DCGAN__ discriminator (D) architecture...

In [8]:
def make_discriminator_dcgan():
    model_layers = [
        Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)),

        # 1st layer: 128 filters without normalization
        layers.Conv2D(128, kernel_size=4, strides=2, padding='same'),
    ]
    if USE_RELU_D == 1:
        model_layers.append(layers.LeakyReLU(negative_slope=0.2))
    if USE_DROPOUT_D == 1:
        model_layers.append(layers.Dropout(DROPOUT_D))

    # 2nd layer: 256 filters
    model_layers.append(layers.Conv2D(256, kernel_size=4, strides=2, padding='same'))
    if NORM_TYPE_D == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    if USE_RELU_D == 1:
        model_layers.append(layers.LeakyReLU(negative_slope=0.2))
    if USE_DROPOUT_D == 1:
        model_layers.append(layers.Dropout(DROPOUT_D))

    # 3rd layer: 512 filters
    model_layers.append(layers.Conv2D(512, kernel_size=4, strides=2, padding='same'))
    if NORM_TYPE_D == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    if USE_RELU_D == 1:
        model_layers.append(layers.LeakyReLU(negative_slope=0.2))
    if USE_DROPOUT_D == 1:
        model_layers.append(layers.Dropout(DROPOUT_D))

    # 4th layer: 1024 filters
    model_layers.append(layers.Conv2D(1024, kernel_size=4, strides=2, padding='same'))
    if NORM_TYPE_D == 'BATCH':
        model_layers.append(layers.BatchNormalization())
    if USE_RELU_D == 1:
        model_layers.append(layers.LeakyReLU(negative_slope=0.2))
    if USE_DROPOUT_D == 1:
        model_layers.append(layers.Dropout(DROPOUT_D))

    # output: 1x1 convolution and flatten
    model_layers.append(layers.Conv2D(1, kernel_size=4, strides=1, padding='valid', activation='sigmoid'))
    model_layers.append(layers.Flatten())

    return keras.Sequential(model_layers)


Once more, let's visualize our model...

In [9]:
# ---------------
# Visualize model
# ---------------

#!pip install visualkeras      # Important!!! Install library in colab, if necessary
import visualkeras
model = make_discriminator_dcgan()
visualkeras.layered_view(model, legend=True)

/usr/local/lib/python3.12/dist-packages/visualkeras/layered.py:231: UserWarning: The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.
  warnings.warn("The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.")


AttributeError: 'Conv2D' object has no attribute 'output_shape'

Let's clean our memory...

In [10]:
del model

## 2.6. Other G-D architectures

This function is ready to implement other well-known GAN architectures...

In [11]:
def make_generator_wgan():
    return make_generator_dcgan()

def make_discriminator_wgan():
    return make_discriminator_dcgan()

def make_generator_pggan():
    return make_generator_dcgan()

def make_discriminator_pggan():
    return make_discriminator_dcgan()

def make_generator_stylegan():
    return make_generator_dcgan()

def make_discriminator_stylegan():
    return make_discriminator_dcgan()

## 2.7. Loss functions

Here let's define some methods used for loss function computation...

In [12]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def wgan_discriminator_loss(real_output, fake_output):
    return tf.reduce_mean(fake_output) - tf.reduce_mean(real_output)

def wgan_generator_loss(fake_output):
    return -tf.reduce_mean(fake_output)


## 2.8. Training

Let's define our training functions...

In [13]:

@tf.function
def train_step(images, generator, discriminator, generator_optimizer, discriminator_optimizer, use_wgan):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        if use_wgan:
            gen_loss = wgan_generator_loss(fake_output)
            disc_loss = wgan_discriminator_loss(real_output, fake_output)
        else:
            gen_loss = generator_loss(fake_output)
            disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Compute gradient norms for visualization
    grad_norm_gen = tf.linalg.global_norm(gradients_of_generator)
    grad_norm_disc = tf.linalg.global_norm(gradients_of_discriminator)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    return gen_loss, disc_loss, generated_images, grad_norm_gen, grad_norm_disc

def train(dataset, generator, discriminator, use_wgan):
    training_history = {
        'gen_loss': [], 'disc_loss': [], 'FID': [],
        'precision': [], 'recall': [],
        'grad_norm_gen': [], 'grad_norm_disc': []
    }
    seed = tf.random.normal([16, NOISE_DIM])

    noise_imgs = tf.random.normal([16, IMAGE_HEIGHT, IMAGE_WIDTH, 1])

    fig = plt.figure(figsize=(4, 4))
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(noise_imgs[i, ..., 0], cmap='gray')
        plt.axis('off')
    plt.suptitle('Random noise images before training')
    plt.show()

    best_fid = float('inf')
    best_epoch = 0
    gen_opt = Adam(LEARNING_RATE_G, beta_1=BETA_1, beta_2=BETA_2)
    disc_opt = Adam(LEARNING_RATE_D, beta_1=BETA_1, beta_2=BETA_2)

    for epoch in range(MAX_EPOCHS):
        #print(f'Starting epoch {epoch+1}/{MAX_EPOCHS}')
        total_batches = tf.data.experimental.cardinality(dataset).numpy()
        prog_bar = tqdm(total=total_batches, desc=f'Epoch {epoch+1}', unit='batch')
        # ... loop setup ...
        for image_batch in dataset:
            for _ in range(NUM_STEPS_D):
                g_loss, d_loss, generated_images, grad_g, grad_d = train_step(image_batch, generator, discriminator, gen_opt, disc_opt, use_wgan)
            for _ in range(NUM_STEPS_G):
                g_loss, d_loss, generated_images, grad_g, grad_d = train_step(image_batch, generator, discriminator, gen_opt, disc_opt, use_wgan)
            prog_bar.update(1)

        # Save all tracked metrics
        fid = compute_fid(image_batch, generated_images)
        prec, rec = compute_precision_recall(image_batch, generated_images)
        training_history['gen_loss'].append(g_loss.numpy())
        training_history['disc_loss'].append(d_loss.numpy())
        training_history['FID'].append(fid)
        training_history['precision'].append(prec)
        training_history['recall'].append(rec)
        training_history['grad_norm_gen'].append(grad_g.numpy())
        training_history['grad_norm_disc'].append(grad_d.numpy())

        if (epoch + 1) % VIS_INTERVAL == 0:
            generate_and_save_images(generator, epoch+1, seed)
            plot_training_metrics(training_history)

        # Save training history to file if enabled
        if SAVE_HISTORY:
            with open('train_history.pkl', 'wb') as f:
                pickle.dump(training_history, f)

        prog_bar.close()

        print("Epoch", epoch+1, ":",
              "G Loss =", g_loss.numpy(),
              ", D Loss =", d_loss.numpy(),
              ", FID =", fid,
              ", Precision =", prec,
              ", Recall =", rec)

        if fid < best_fid:
            best_fid = fid
            best_epoch = epoch
            generator.save('best_generator.keras')
            discriminator.save('best_discriminator.keras')

        if epoch - best_epoch > EARLY_STOP_PATIENCE:
            print("Early stopping due to FID")
            break

    return training_history

## 2.9. Main loop

Now let's finally run our GAN...


In [14]:
# ----------------
# GAN main loop
# ----------------

# Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

# Compute BCE
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# Load the inception model
inception_model = tf.keras.applications.InceptionV3(
    include_top=False, pooling='avg', input_shape=(299, 299, 3)
)

# if data not already loaded, load it.
if 'celeba_dataset' not in globals():
    celeba_dataset = load_celeba_images(ZIP_PATH, max_images=NUM_IMAGES)
else:
    print('Real images dataset previously loaded.')


if MODEL_TYPE == 'DCGAN':
    generator = make_generator_dcgan()
    discriminator = make_discriminator_dcgan()
    use_wgan = False
elif MODEL_TYPE == 'WGAN':
    generator = make_generator_wgan()
    discriminator = make_discriminator_wgan()
    use_wgan = True
elif MODEL_TYPE == 'PGGAN':
    generator = make_generator_pggan()
    discriminator = make_discriminator_pggan()
    use_wgan = False
elif MODEL_TYPE == 'StyleGAN':
    generator = make_generator_stylegan()
    discriminator = make_discriminator_stylegan()
    use_wgan = False
if LOAD_BEST_MODEL:
    generator = load_model('best_generator.keras')
    discriminator = load_model('best_discriminator.keras')
    print("Best model loaded from file.")

history = train(celeba_dataset, generator, discriminator, use_wgan)


Mounted at /content/drive
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/img_align_celeba.zip'

## 2.10. View history

In case you need to improve your GANs, here is a code to print __training history__...

In [15]:
# --------------------------
# Print training history log
# --------------------------

import pickle

# Load the training history from the file
with open('train_history.pkl', 'rb') as f:
    training_history = pickle.load(f)

# Print the training history
print("Training History:")
for key, values in training_history.items():
    print(f"{key}:")
    print(values)
print("\n")

FileNotFoundError: [Errno 2] No such file or directory: 'train_history.pkl'

##  2.11. Generating faces

Finally, let's generate some celebrity faces using our trained GAN...


In [16]:
# ------------------------------------------------------------
# Generate new images using the best generator model saved
# ------------------------------------------------------------

import os
import matplotlib.pyplot as plt
from keras.models import load_model

NUM_SAMPLES = 16  # Must be a square number (e.g., 4×4) for grid display
FIG_SIZE = 6      # Size of output image

# Path to the saved generator model (automatically saved during training)
GENERATOR_PATH = 'best_generator.keras'

# Check if the model file exists
if os.path.exists(GENERATOR_PATH):
    print("Loading the best saved generator model...")

    # Load the trained generator model
    best_generator = load_model(GENERATOR_PATH)

    # Create random noise vectors as input (latent space samples)
    seed_noise = tf.random.normal([NUM_SAMPLES, NOISE_DIM])

    # Generate images using the loaded generator (inference mode)
    generated_images = best_generator(seed_noise, training=False)

    # Plot the generated images in a 4×4 grid
    plt.figure(figsize=(FIG_SIZE, FIG_SIZE))
    for i in range(NUM_SAMPLES):
        plt.subplot(4, 4, i + 1)

        # Convert image pixel values from [-1, 1] → [0, 255] for display
        img = (generated_images[i] * 127.5 + 127.5).numpy().astype(np.uint8)

        plt.imshow(img)
        plt.axis('off')

    plt.suptitle("Generated Images from Best Generator (Lowest FID)", fontsize=12)
    plt.tight_layout()
    plt.show()

else:
    print("'best_generator.keras' not found. Please train the model first.")


'best_generator.keras' not found. Please train the model first.


____

# 3. Next steps

* __Questions__:
  * Considering the CelebA dataset, tha faces generated by our GAN have any kind of bias?
* __Further implementations__:
  * In our training we are using only __a fraction of the number of faces available__ in the file. Try to use more faces and evaluate the quality of the final images generated...
  * We adopted a resolution of 64x64. Explore more complex architectures that can generate **higher quality images**!
  * Not all GAN architectures are implemented in this code. You can try to __implement and test other models like PGGAN, WGAN, StyleGAN__...
<br>
<br>

____

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/0/0a/Logo_Unesp.svg" width="400" style="float: left; margin-right: 5px;" border="0px" />
</center>